In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [31]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq 
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

In [41]:
loader = PyPDFLoader("../pdf/medical_report.pdf")
docs = loader.load()
len(docs)

9

In [42]:
splitter = RecursiveCharacterTextSplitter(chunk_size= 1000 , chunk_overlap=200)
splited_data= splitter.split_documents(docs)
len(splited_data)

26

In [43]:
embedding = OllamaEmbeddings(model="all-minilm")

In [44]:
vector_store = InMemoryVectorStore.from_documents(
    documents = splited_data,
    embedding= embedding
)

In [45]:
same_record= vector_store.similarity_search("Student name")

In [14]:
# len(same_record)
# print(same_record[1].page_content)

agent = tool , llm , prompt

In [46]:
@tool
def retriver_tool(query:str):
    """This tool can help you to retriver the relevant date form the document and this doc have details about internship report"""
    print("Tool called :", query)
    docs  = vector_store.similarity_search(query=query , k=4)
    context = ""
    for doc in docs :
        context = doc.page_content + "\n\n"

    return context

In [28]:
retriver_tool.invoke("Student name")

'D. Y. PATIL TECHNICAL CAMPUS, \nFACULTY OF ENGINEERING & FACULTY OF MANAGEMENT, TALSANDE \nDepartment of Computer Science & Engineering (Academic 2026-27) \nWeekly Internship Report \nStudent Name: Smartha Shesherao Radkar Roll No.:  \nCompany Name: Codexlabz Technologies, Atpadi, Sangli Place: Sangli \nWeek Duration: From: 15 June 2026   To: 20 June 2026 Reporting Week No.: 2 \nCompany Mentor: Ganesh Sutar \n \nWork Summary (Tasks/Activities Performed During the Week): \nSr. \nNo. Date Task / Activity Performed \n1 15 June Introduction to databases, DBMS, tables, rows, columns and primary keys \n2 16 June Practiced SQL SELECT statements and filtering using WHERE conditions \n3 17 June Practiced ORDER BY, DISTINCT, LIMIT and different SQL operators \n4 18 June Worked with aggregate functions such as COUNT, SUM, AVG, MIN and MAX \n5 19 June Practiced GROUP BY and HAVING for analyzing grouped data \n6 20 June Performed SQL joins and created queries to extract meaningful information from

In [47]:
llm= ChatGroq(model="openai/gpt-oss-20b")

In [48]:
system_prompt= """You are a assistantthat answers questions using retrived context
                always use `retriver_tool` tool for question requiring external knowledge """

In [49]:
agent = create_agent(
    model = llm , 
    tools= [retriver_tool],
    system_prompt= system_prompt
)

In [50]:
query = "What is name of patient and what is name of doctors"
res = agent.invoke({"messages": [{"role": "user", "content": query}]}) 

Tool called : patient name and doctors names in internship report


In [51]:
result = res["messages"][-1].content
print(result)

**Patient:** Ms. Nikita Chudhary  

**Doctors mentioned in the report:**  
- Dr. Nitin Nahar  
- Dr. Swapnil Gupta, MD, Pathology (Chief of Laboratory, Dr Lal PathLabs Ltd)  
- Dr. Kiran Bhargava Pathak, MD, Pathology (Chief of Laboratory, Dr Lal PathLabs Ltd)
